In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch


# ----------------------------
# Project structure
# ----------------------------
ROOT = Path(r"C:\Users\am636\copperbelt_worldpop_smod")

DATA = ROOT / "data_raw"
OUT = ROOT / "outputs"
FIGDIR = ROOT / "figures"
TABDIR = ROOT / "tables"

for d in (DATA, OUT, FIGDIR, TABDIR):
    d.mkdir(parents=True, exist_ok=True)

PREFIX = "task1_"

adm2_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
pop_path = DATA / "zmb_ppp_2020_constrained.tif"
smod_path = DATA / "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif"

missing = [p for p in (adm2_path, pop_path, smod_path) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required input file(s) in data_raw:\n" + "\n".join(str(p) for p in missing)
    )

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# ----------------------------
# Metadata helpers
# ----------------------------
def raster_meta(path: Path) -> dict:
    with rasterio.open(path) as src:
        return {
            "path": str(path),
            "crs": str(src.crs),
            "width": int(src.width),
            "height": int(src.height),
            "bands": int(src.count),
            "dtype": list(src.dtypes),
            "nodata": src.nodata,
            "pixel_size": tuple(float(x) for x in src.res),
            "bounds": {
                "left": float(src.bounds.left),
                "bottom": float(src.bounds.bottom),
                "right": float(src.bounds.right),
                "top": float(src.bounds.top),
            },
            "driver": src.driver,
        }


def vector_meta(path: Path) -> dict:
    gdf = gpd.read_file(path)
    minx, miny, maxx, maxy = (float(x) for x in gdf.total_bounds)
    out = {
        "path": str(path),
        "crs": str(gdf.crs),
        "feature_count": int(len(gdf)),
        "geometry_type": str(gdf.geom_type.iloc[0]) if len(gdf) else None,
        "columns": list(gdf.columns),
        "bounds": {"minx": minx, "miny": miny, "maxx": maxx, "maxy": maxy},
    }
    if "NAME_2" in gdf.columns:
        out["NAME_2_sample"] = gdf["NAME_2"].head(10).tolist()
    return out


# ----------------------------
# Window read around Copperbelt
# ----------------------------
def read_window_for_polygon(raster_path: Path, poly_gdf: gpd.GeoDataFrame, pad: float = 0.0):
    """
    Reads a single-band raster window around the polygon bounds.
    pad is in raster CRS units (degrees for EPSG:4326, metres for Mollweide).
    """
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        nodata = src.nodata

        gdf_r = poly_gdf.to_crs(r_crs)
        minx, miny, maxx, maxy = gdf_r.total_bounds
        minx -= pad
        miny -= pad
        maxx += pad
        maxy += pad

        win = from_bounds(minx, miny, maxx, maxy, transform=src.transform)
        win = win.round_offsets().round_lengths()

        arr = src.read(1, window=win)
        transform = src.window_transform(win)

        return arr, transform, r_crs, nodata


def extent_from_transform(transform, width: int, height: int):
    left, top = transform * (0, 0)
    right, bottom = transform * (width, height)
    return [left, right, bottom, top]


# ----------------------------
# Task 1: metadata artefacts
# ----------------------------
record = {
    "run_time_utc": run_time_utc,
    "project_root": str(ROOT),
    "folders": {
        "data_raw": str(DATA),
        "outputs": str(OUT),
        "figures": str(FIGDIR),
        "tables": str(TABDIR),
    },
    "files": {
        "adm2": str(adm2_path),
        "worldpop": str(pop_path),
        "smod": str(smod_path),
    },
    "layers": {
        "adm2": vector_meta(adm2_path),
        "worldpop": raster_meta(pop_path),
        "smod": raster_meta(smod_path),
    },
}

json_path = OUT / f"{PREFIX}00_sanity_check_metadata.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

rows = []
for layer_name, meta in record["layers"].items():
    if layer_name == "adm2":
        rows.append(
            {
                "layer": "adm2",
                "type": "vector",
                "crs": meta["crs"],
                "feature_count": meta["feature_count"],
                "width": None,
                "height": None,
                "pixel_x": None,
                "pixel_y": None,
                "bands": None,
                "dtype": None,
                "nodata": None,
                "bounds": f'{meta["bounds"]["minx"]},{meta["bounds"]["miny"]},{meta["bounds"]["maxx"]},{meta["bounds"]["maxy"]}',
                "path": meta["path"],
            }
        )
    else:
        rows.append(
            {
                "layer": layer_name,
                "type": "raster",
                "crs": meta["crs"],
                "feature_count": None,
                "width": meta["width"],
                "height": meta["height"],
                "pixel_x": meta["pixel_size"][0],
                "pixel_y": meta["pixel_size"][1],
                "bands": meta["bands"],
                "dtype": ";".join(meta["dtype"]),
                "nodata": meta["nodata"],
                "bounds": f'{meta["bounds"]["left"]},{meta["bounds"]["bottom"]},{meta["bounds"]["right"]},{meta["bounds"]["top"]}',
                "path": meta["path"],
            }
        )

csv_path = TABDIR / f"{PREFIX}00_sanity_check_layers.csv"
pd.DataFrame(rows).to_csv(csv_path, index=False)


# ----------------------------
# Task 1: figure (WorldPop + SMOD + ADM2)
# ----------------------------
adm2 = gpd.read_file(adm2_path)

# WorldPop (EPSG:4326): pad in degrees
pop_arr, pop_tr, pop_crs, pop_nodata = read_window_for_polygon(pop_path, adm2, pad=0.15)
pop_masked = np.ma.masked_where(pop_arr == pop_nodata, pop_arr) if pop_nodata is not None else np.ma.masked_invalid(pop_arr)
pop_display = np.ma.log1p(pop_masked)
pop_extent = extent_from_transform(pop_tr, pop_display.shape[1], pop_display.shape[0])
adm2_pop = adm2.to_crs(pop_crs)

# SMOD (Mollweide): pad in metres
smod_arr, smod_tr, smod_crs, smod_nodata = read_window_for_polygon(smod_path, adm2, pad=25_000)
smod_mask = np.zeros_like(smod_arr, dtype=bool)
if smod_nodata is not None:
    smod_mask |= (smod_arr == smod_nodata)
smod_mask |= (smod_arr == 0)
smod_masked = np.ma.masked_where(smod_mask, smod_arr)
smod_extent = extent_from_transform(smod_tr, smod_arr.shape[1], smod_arr.shape[0])
adm2_smod = adm2.to_crs(smod_crs)

# Categorical colours (class 2 is blue)
cmap = ListedColormap(
    [
        (0.85, 0.85, 0.85, 1.0),  # background
        (0.45, 0.75, 0.45, 1.0),  # 1 rural
        (0.20, 0.40, 0.90, 1.0),  # 2 urban cluster (blue)
        (0.90, 0.45, 0.20, 1.0),  # 3 urban centre
    ]
)
norm = BoundaryNorm([0, 1, 2, 3, 4], cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
im0 = ax.imshow(pop_display, extent=pop_extent, origin="upper")
adm2_pop.boundary.plot(ax=ax, linewidth=1.2, color="black")
ax.set_title("WorldPop 2020 constrained (log-scaled) + Copperbelt ADM2")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar0 = plt.colorbar(im0, ax=ax, fraction=0.046, pad=0.04)
cbar0.set_label("log(1 + population)")

ax = axes[1]
ax.imshow(smod_masked, extent=smod_extent, origin="upper", cmap=cmap, norm=norm, interpolation="nearest")
adm2_smod.boundary.plot(ax=ax, linewidth=1.2, color="black")
ax.set_title("GHS-SMOD (categorical) + Copperbelt ADM2")
ax.set_xlabel("X (metres)")
ax.set_ylabel("Y (metres)")
ax.legend(
    handles=[
        Patch(facecolor=cmap(1), edgecolor="black", label="1 = Rural"),
        Patch(facecolor=cmap(2), edgecolor="black", label="2 = Urban cluster (blue)"),
        Patch(facecolor=cmap(3), edgecolor="black", label="3 = Urban centre"),
    ],
    loc="lower left",
    frameon=True,
)

plt.tight_layout()
map_path = FIGDIR / f"{PREFIX}00_layers_map.png"
plt.savefig(map_path, dpi=200, bbox_inches="tight")
plt.show()


print("\nSaved artefacts:")
print("JSON  ->", json_path)
print("CSV   ->", csv_path)
print("FIG   ->", map_path)


Run time (UTC): 2026-01-24T20:31:52+00:00
ROOT exists: True
DATA exists: True
OUT exists : True

Files exist: {'adm2': True, 'worldpop': True, 'smod': True}

ADM2: 12 features | CRS: EPSG:4326
WorldPop: CRS: EPSG:4326 | dims: (14047, 11826) | res: (0.0008333333300348827, 0.000833333330035515) | nodata: -99999.0
SMOD: CRS: ESRI:54009 | dims: (1000, 1000) | res: (1000.0, 1000.0) | nodata: -200.0

Saved:
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\00_sanity_check_metadata.json
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\00_sanity_check_layers.csv
